In [1]:
# CELL 1: Imports & global configuration

import os
import json
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


In [2]:
# CELL 2: Load OpenAI API key

with open("key.txt", "r", encoding="utf-8") as f:
    OPENAI_API_KEY = f.read().strip()

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


In [3]:
# CELL 3: Load datasets (sentence-level classification)

majaz = pd.read_csv("majaz_dataset.csv")
non_majaz = pd.read_csv("non_majaz_random_sample.csv")

majaz = majaz.copy()
non_majaz = non_majaz.copy()

majaz["label"] = 1
non_majaz["label"] = 0

df = pd.concat([majaz, non_majaz], ignore_index=True)

TEXT_COL = "البيت_الكامل"

texts_raw = df[TEXT_COL].astype(str).tolist()
y = df["label"].values

print(df["label"].value_counts())


label
1    842
0    842
Name: count, dtype: int64


In [4]:
# CELL 4: Arabic diacritics removal

ARABIC_DIACRITICS = re.compile(
    r"[ًٌٍَُِّْـ]"
)

def normalize_ar(text: str) -> str:
    return re.sub(ARABIC_DIACRITICS, "", text)

texts = [normalize_ar(t) for t in texts_raw]

print(texts_raw[0])
print(texts[0])


مَرَرْتُ عَلى دارِ الحَبِيبِ فحَمْحَمَتْ جَوَادي وَهَلْ تَشْجو الجِيادَ المَعاهِدُ
مررت على دار الحبيب فحمحمت جوادي وهل تشجو الجياد المعاهد


In [5]:
# CELL 5: CAMeL Tools analyzer

from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer
from camel_tools.tokenizers.word import simple_word_tokenize

db = MorphologyDB.builtin_db()
analyzer = Analyzer(db, backoff="NOAN_PROP")

print(analyzer.analyze("تشكو")[0])


{'diac': 'تَشْكُو', 'lex': 'شَكا', 'bw': 'تَ/IV2MS+شْكُو/IV', 'gloss': 'you_[masc.sg.]+complain;suffer', 'pos': 'verb', 'prc3': '0', 'prc2': '0', 'prc1': '0', 'prc0': '0', 'per': '2', 'asp': 'i', 'vox': 'a', 'mod': 'u', 'stt': 'na', 'cas': 'na', 'enc0': '0', 'rat': 'n', 'source': 'lex', 'form_gen': 'm', 'form_num': 's', 'd3seg': 'تَشْكُو', 'caphi': 't_a_sh_k_u_w', 'd1tok': 'تَشْكُو', 'd2tok': 'تَشْكُو', 'pos_logprob': -1.023208, 'd3tok': 'تَشْكُو', 'd2seg': 'تَشْكُو', 'pos_lex_logprob': -4.286608, 'num': 's', 'ud': 'VERB', 'gen': 'm', 'catib6': 'VRB', 'root': 'ش.ك.#', 'bwtok': 'تَ+_شْكُو', 'pattern': 'تَ1ْ2ُو', 'lex_logprob': -4.286608, 'atbtok': 'تَشْكُو', 'atbseg': 'تَشْكُو', 'd1seg': 'تَشْكُو', 'stem': 'شْكُو', 'stemgloss': 'complain;suffer', 'stemcat': 'IV_0hAnn'}


In [6]:
# CELL 6: Symbolic linguistic features

def symbolic_features(text):
    tokens = simple_word_tokenize(text)

    verbs = nouns = passive = animacy_clash = 0
    roots = []

    for i, tok in enumerate(tokens):
        a = analyzer.analyze(tok)
        if not a:
            continue
        a = a[0]

        if a.get("ud") == "VERB":
            verbs += 1
            if a.get("vox") == "p":
                passive += 1

        if a.get("ud") == "NOUN":
            nouns += 1

        roots.append(a.get("root"))

        # agent–verb animacy clash (noun → verb)
        if i < len(tokens) - 1:
            n = analyzer.analyze(tokens[i])
            v = analyzer.analyze(tokens[i + 1])
            if n and v:
                if n[0].get("ud") == "NOUN" and v[0].get("ud") == "VERB":
                    if n[0].get("pattern", "").startswith(("فِع", "فَع", "مَف")):
                        animacy_clash += 1

    n = max(len(tokens), 1)
    return {
        "verb_ratio": verbs / n,
        "noun_ratio": nouns / n,
        "passive_ratio": passive / n,
        "animacy_clash": animacy_clash,
        "root_repetition": 1 - len(set(roots)) / max(len(roots), 1)
    }


In [7]:
# CELL 7: Symbolic feature matrix

X_symbolic = pd.DataFrame(
    [symbolic_features(t) for t in tqdm(texts, desc="Symbolic features")]
)

print(X_symbolic.head())


Symbolic features: 100%|██████████| 1684/1684 [00:41<00:00, 40.45it/s]

   verb_ratio  noun_ratio  passive_ratio  animacy_clash  root_repetition
0    0.200000    0.300000            0.0              0             0.10
1    0.100000    0.500000            0.0              0             0.10
2    0.000000    0.000000            0.0              0             0.00
3    0.416667    0.333333            0.0              0             0.25
4    0.000000    0.500000            0.0              0             0.25


In [8]:
# CELL 8: Arabic sentence embeddings (local)

from sentence_transformers import SentenceTransformer

arabic_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

X_embed_ar = arabic_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(X_embed_ar.shape)


C:\Users\USER\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\huggingface_hub\utils\_validators.py:186: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  warnings.warn(
Batches: 100%|██████████| 53/53 [00:00<00:00, 58.27it/s]

(1684, 384)


In [9]:
# CELL 9: GPT large embeddings

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

def gpt_embed(text):
    emb = client.embeddings.create(
        model="text-embedding-3-large",
        input=text
    )
    return np.array(emb.data[0].embedding, dtype=np.float32)

X_embed_gpt = np.vstack([
    gpt_embed(t) for t in tqdm(texts, desc="GPT embeddings")
])

print(X_embed_gpt.shape)


GPT embeddings: 100%|██████████| 1684/1684 [14:26<00:00,  1.94it/s]

(1684, 3072)


In [10]:
# CELL 10 (FIXED): GPT diagnostic reasoning features

import json
import re

def gpt_reason_features(text):
    prompt = f"""
أنت خبير في البلاغة العربية.
أجب فقط بصيغة JSON صالحة بدون أي شرح.

البيت:
{text}

أعد JSON فقط بهذا الشكل EXACTLY:

{{
  "logical_violation": 0,
  "implicit_agent": 0,
  "literal_acceptability": 1
}}

القيم يجب أن تكون أرقام 0 أو 1 فقط.
"""

    resp = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "You are a JSON-only generator."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    content = resp.choices[0].message.content.strip()

    # Try direct JSON parse
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        # Fallback: extract JSON block via regex
        match = re.search(r"\{.*\}", content, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass

    # Final fallback (safe default, NEVER crash)
    return {
        "logical_violation": 0,
        "implicit_agent": 0,
        "literal_acceptability": 1
    }

# CELL 10b: Build GPT reasoning feature matrix (safe)

X_gpt_reason = pd.DataFrame([
    gpt_reason_features(t) for t in tqdm(texts, desc="GPT reasoning")
])

print(X_gpt_reason.head())


GPT reasoning: 100%|██████████| 1684/1684 [26:45<00:00,  1.05it/s]

   logical_violation  implicit_agent  literal_acceptability
0                  0               1                      1
1                  1               1                      0
2                  0               1                      1
3                  1               1                      0
4                  0               0                      1


In [11]:
# CELL 11: Feature concatenation

X_all = np.hstack([
    X_symbolic.values,
    X_embed_ar,
    X_embed_gpt,
    X_gpt_reason.values
])

print(X_all.shape)


(1684, 3464)


In [12]:
# CELL 12: Stratified random split (sentence-level)

X_train_idx, X_test_idx, y_train, y_test = train_test_split(
    np.arange(len(y)),
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

def split(X):
    return X[X_train_idx], X[X_test_idx]

print("Train samples:", len(X_train_idx))
print("Test samples:", len(X_test_idx))


Train samples: 1178
Test samples: 506


In [13]:
# CELL 13: Unified classifier

Xtr, Xte = split(X_all)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=3000))
])

pipeline.fit(Xtr, y_train)
y_pred = pipeline.predict(Xte)

print(classification_report(y_test, y_pred, digits=3))


              precision    recall  f1-score   support

           0      0.533     0.510     0.521       253
           1      0.530     0.553     0.542       253

    accuracy                          0.532       506
   macro avg      0.532     0.532     0.531       506
weighted avg      0.532     0.532     0.531       506



In [14]:
# CELL 14: One prediction function (final model)

def predict_majaz(verse: str):
    verse_n = normalize_ar(verse)

    S = pd.DataFrame([symbolic_features(verse_n)]).values
    E_ar = arabic_model.encode([verse_n], normalize_embeddings=True)
    E_gpt = gpt_embed(verse_n).reshape(1, -1)
    G = pd.DataFrame([gpt_reason_features(verse_n)]).values

    X = np.hstack([S, E_ar, E_gpt, G])
    return pipeline.predict(X)[0]


In [15]:
# CELL 15: Evaluation helper for ablations

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

def eval_model(X_train, X_test, y_train, y_test, name):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=3000))
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, digits=3))

    return acc


In [16]:
# CELL 16: Feature blocks

X_sym   = X_symbolic.values          # CAMeL symbolic features
X_ar    = X_embed_ar                 # Arabic sentence embeddings
X_gpt_e = X_embed_gpt                # GPT embeddings
X_gpt_r = X_gpt_reason.values        # GPT reasoning features

# Combinations
X_sym_ar      = np.hstack([X_sym, X_ar])
X_sym_ar_gpte = np.hstack([X_sym, X_ar, X_gpt_e])
X_sym_ar_gpter= np.hstack([X_sym, X_ar, X_gpt_r])
X_all         = np.hstack([X_sym, X_ar, X_gpt_e, X_gpt_r])


In [17]:
# CELL 18: Ablation experiments (same split everywhere)

def eval_model(X, name):
    Xtr, Xte = split(X)

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=3000))
    ])

    pipe.fit(Xtr, y_train)
    y_pred = pipe.predict(Xte)

    acc = accuracy_score(y_test, y_pred)

    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, digits=3))

    return acc


results = {}
results["Symbolic only"] = eval_model(X_sym, "Symbolic only")
results["Arabic embeddings only"] = eval_model(X_ar, "Arabic embeddings only")
results["GPT embeddings only"] = eval_model(X_gpt_e, "GPT embeddings only")
results["GPT reasoning only"] = eval_model(X_gpt_r, "GPT reasoning only")
results["ALL FEATURES"] = eval_model(X_all, "ALL FEATURES")



=== Symbolic only ===
Accuracy: 0.5553
              precision    recall  f1-score   support

           0      0.558     0.530     0.544       253
           1      0.553     0.581     0.566       253

    accuracy                          0.555       506
   macro avg      0.555     0.555     0.555       506
weighted avg      0.555     0.555     0.555       506


=== Arabic embeddings only ===
Accuracy: 0.5000
              precision    recall  f1-score   support

           0      0.500     0.486     0.493       253
           1      0.500     0.514     0.507       253

    accuracy                          0.500       506
   macro avg      0.500     0.500     0.500       506
weighted avg      0.500     0.500     0.500       506


=== GPT embeddings only ===
Accuracy: 0.5316
              precision    recall  f1-score   support

           0      0.533     0.518     0.525       253
           1      0.531     0.545     0.538       253

    accuracy                          0.532    